# Introduction to Vector Databases (Installing and Running ChromaDB)

**Level:** Scratch → Intermediate
**Runs on:** Google Colab (Free CPU)

### What you will learn
1. What a Vector Database is and why we need one
2. Installing ChromaDB (free, open-source, runs locally)
3. Creating a Collection (like a table, but for vectors)
4. Adding documents (ChromaDB auto-generates embeddings)
5. Querying by meaning (semantic search)
6. Using your own Hugging Face embeddings with ChromaDB (intermediate)
7. Updating and deleting records

> No API key, no cloud account, no cost. Everything runs in-memory inside this notebook.

## 1. Why a Vector Database?

A normal database (SQL) searches by **exact match** — e.g., `WHERE name = 'dog'`.

A **Vector Database** stores embeddings (numeric meaning-vectors) and searches by **similarity** — e.g., "find text that *means* something like 'dog'", even if the word "dog" never appears.

| Normal Database | Vector Database |
|---|---|
| Matches exact text/values | Matches by meaning |
| Uses SQL queries | Uses similarity search |
| Good for structured data | Good for text, images, audio embeddings |

**ChromaDB** is one of the simplest free, open-source vector databases — great for learning.

### 2. Install ChromaDB
Run this once. `-q` keeps the output clean.

In [16]:
!pip install chromadb sentence-transformers

In [17]:
!pip install -q chromadb

### 3. Create a ChromaDB Client

We use an **in-memory / ephemeral client** — perfect for learning in Colab.
(Data will reset when the runtime restarts. For permanent storage, `chromadb.PersistentClient(path="...")` is used instead — shown later.)

In [18]:
import chromadb

client = chromadb.Client()

print("ChromaDB client created successfully!")

ChromaDB client created successfully!


### 4. Create a Collection

A **Collection** in ChromaDB is like a table — it stores your documents, their embeddings, and metadata.

In [20]:
collection = client.get_or_create_collection(name="my_first_collection")

print("Collection created or retrieved:", collection.name)

Collection created or retrieved: my_first_collection


### 5. Add Documents

We simply give ChromaDB plain text. By default, it **automatically converts text into embeddings** for us using a built-in free model — no extra setup needed.

In [21]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers.",
        "Paris is the capital city of France."
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6"]
)

print("Documents added! Total documents:", collection.count())

Documents added! Total documents: 8


**Discussion Point:** Every document got an `id` (like a primary key). Behind the scenes, ChromaDB converted each sentence into an embedding vector and stored it — just like we did manually in the earlier embeddings notebook.

### 6. Query by Meaning (Semantic Search)

Now we ask a question in natural language. ChromaDB converts our query into an embedding too, then finds the closest matching documents.

In [22]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.9317  |  Deep learning uses neural networks with many layers.
Distance: 0.9516  |  Python is a popular programming language for AI.


**Discussion Point:** Lower `distance` = more similar (closer meaning). Notice the AI/ML related sentences come back first, even though the query never used the exact same words.

In [23]:
# Try another query
results = collection.query(
    query_texts=["What is the capital of France?"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.3515  |  Paris is the capital city of France.
Distance: 1.7959  |  Python is a popular programming language for AI.


### 7. Adding Metadata (Intermediate)

Real-world documents usually have extra info (category, author, date). ChromaDB lets us store this alongside each document and filter on it.

In [24]:
collection.add(
    documents=[
        "The new smartphone has an amazing camera.",
        "Stock prices rose sharply after the earnings report."
    ],
    metadatas=[
        {"category": "technology"},
        {"category": "finance"}
    ],
    ids=["doc7", "doc8"]
)

# Query only within the "technology" category
results = collection.query(
    query_texts=["Tell me about new gadgets"],
    n_results=2,
    where={"category": "technology"}
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 1.1460  |  The new smartphone has an amazing camera.


### 8. Using Your Own Hugging Face Embeddings (Intermediate)

By default ChromaDB uses its own built-in embedding model. But we can plug in **any** Hugging Face model, such as `all-MiniLM-L6-v2` from the earlier notebook, for full control.

In [25]:
!pip install -q sentence-transformers

In [28]:
from chromadb.utils import embedding_functions

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create or retrieve a collection that uses our chosen Hugging Face model
custom_collection = client.get_or_create_collection(
    name="custom_embedding_collection",
    embedding_function=hf_embedder
)

custom_collection.add(
    documents=[
        "I love dogs",
        "I like puppies",
        "The stock market crashed today"
    ],
    ids=["a1", "a2", "a3"]
)

results = custom_collection.query(
    query_texts=["puppy lovers"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.2513  |  I like puppies
Distance: 0.3943  |  I love dogs


### 9. Update and Delete Records

In [29]:
# Update a document's text
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."]
)
print("Updated doc4")

# Delete a document
collection.delete(ids=["doc2"])
print("Deleted doc2")

print("Remaining documents:", collection.count())

Updated doc4
Deleted doc2
Remaining documents: 7


### 10. Persistent Storage (So Data Survives a Restart)

The examples above used an in-memory client, which resets on restart. For real projects, use `PersistentClient` to save data to disk.

In [30]:
persistent_client = chromadb.PersistentClient(path="/content/chroma_db")

persistent_collection = persistent_client.get_or_create_collection(name="persistent_demo")

persistent_collection.add(
    documents=["This data will be saved to disk."],
    ids=["p1"]
)

print("Saved to disk at /content/chroma_db")
print("Count:", persistent_collection.count())

Saved to disk at /content/chroma_db
Count: 1


### 11. Key Takeaways (Recap for Students)

| Concept | Meaning |
|---|---|
| Vector Database | Stores embeddings, searches by meaning/similarity |
| ChromaDB | Free, open-source, runs locally — no cloud needed |
| Collection | Like a table — holds documents + embeddings + metadata |
| `add()` | Insert documents (embeddings created automatically) |
| `query()` | Search by meaning, returns closest matches |
| `where={}` | Filter results using metadata |
| `PersistentClient` | Saves the database to disk instead of memory |

### 12. Practice Exercises
1. Add 5 of your own sentences to a new collection and query them with a related question.
2. Add a `"category"` metadata field to every document, then filter a query by two different categories.
3. Swap the embedding function to `'all-mpnet-base-v2'` and compare the distance scores to `all-MiniLM-L6-v2`.

In [31]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="exercise1"
)

collection.upsert(
    ids=["1", "2", "3", "4", "5"],
    documents=[
        "Cloud computing provides on-demand computing resources.",
        "Cybersecurity protects systems from digital attacks.",
        "Data analytics helps organizations make informed decisions.",
        "Mobile app development is popular among developers.",
        "Artificial intelligence powers modern virtual assistants."
    ]
)

query = "How do virtual assistants become intelligent?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("\nTop 3 Results:")

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

Query: How do virtual assistants become intelligent?

Top 3 Results:
Document: Artificial intelligence powers modern virtual assistants.
Distance: 0.6161

Document: Cloud computing provides on-demand computing resources.
Distance: 1.597

Document: Cybersecurity protects systems from digital attacks.
Distance: 1.6603



In [32]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="exercise2"
)

collection.upsert(
    ids=["1", "2", "3", "4", "5", "6"],
    documents=[
        "React is used for frontend web development.",
        "Node.js is used for backend applications.",
        "Tennis is played with a racket.",
        "Swimming improves physical fitness.",
        "Deep learning uses neural networks.",
        "Data science combines statistics and programming."
    ],
    metadatas=[
        {"category": "development"},
        {"category": "development"},
        {"category": "sports"},
        {"category": "sports"},
        {"category": "AI"},
        {"category": "AI"}
    ]
)

results = collection.query(
    query_texts=["How can I build websites?"],
    n_results=2,
    where={"category": "development"}
)

print("Development Results:")

for document in results["documents"][0]:
    print("-", document)

results = collection.query(
    query_texts=["Tell me about fitness activities"],
    n_results=2,
    where={"category": "sports"}
)

print("\nSports Results:")

for document in results["documents"][0]:
    print("-", document)

Development Results:
- React is used for frontend web development.
- Node.js is used for backend applications.

Sports Results:
- Swimming improves physical fitness.
- Tennis is played with a racket.


In [34]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_db")

documents = [
    "Cybersecurity protects computer networks from attacks.",
    "Artificial intelligence enables smart automation.",
    "Cloud computing offers scalable resources.",
    "Blockchain provides secure digital transactions.",
    "Data science extracts insights from data."
]

ids = ["1", "2", "3", "4", "5"]

minilm = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

mpnet = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

mini_collection = client.get_or_create_collection(
    name="mini_compare",
    embedding_function=minilm
)

mpnet_collection = client.get_or_create_collection(
    name="mpnet_compare",
    embedding_function=mpnet
)

mini_collection.upsert(
    ids=ids,
    documents=documents
)

mpnet_collection.upsert(
    ids=ids,
    documents=documents
)
query = "How can machines perform tasks automatically?"

mini_results = mini_collection.query(
    query_texts=[query],
    n_results=3
)

mpnet_results = mpnet_collection.query(
    query_texts=[query],
    n_results=3
)

print("===== all-MiniLM-L6-v2 =====")

for document, distance in zip(
    mini_results["documents"][0],
    mini_results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

print("===== all-mpnet-base-v2 =====")

for document, distance in zip(
    mpnet_results["documents"][0],
    mpnet_results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

===== all-MiniLM-L6-v2 =====
Document: Artificial intelligence enables smart automation.
Distance: 0.4896

Document: Cloud computing offers scalable resources.
Distance: 0.7509

Document: Data science extracts insights from data.
Distance: 0.8197

===== all-mpnet-base-v2 =====
Document: Artificial intelligence enables smart automation.
Distance: 0.4636

Document: Data science extracts insights from data.
Distance: 0.7508

Document: Cloud computing offers scalable resources.
Distance: 0.7686

